# 11. CloudXeus Orders API — Azure Function App (Anonymous Auth)

This is an **Azure Functions** app (Python v2 programming model, `azure.functions`), not a Foundry script — it
implements a small REST API (`GET /orders`, `GET /orders/{order_id}`) backed by an in-memory dictionary of
sample orders. Alongside it, `cloudxeus_spec.json` is an **OpenAPI 3.0.3 specification** describing these same
two endpoints — this is exactly the artifact Azure AI Foundry needs to register this Function App as an
**OpenAPI-based custom tool** for an agent (an "action"), letting an agent answer questions like "what's the
status of order 1002?" by calling this real HTTP API instead of a `FunctionTool` your own client code executes
(contrast with `05_IT_HelpDesk_agent.py`/`07_helpdesk_client.py`).

This version's routes use `auth_level=func.AuthLevel.ANONYMOUS` — no authentication required, appropriate for
local development but not for a real deployment (see `12. cloudxeus-func1/function_app.py`, the security-
hardened variant, for the fix).

A notebook cannot literally *host* an HTTP-triggered function (there's no server process backing a notebook
cell). This notebook instead: (a) shows the real, correct function code in runnable cells, and (b) explains how
you'd actually run and test it locally with the Azure Functions Core Tools.

**Difficulty:** Intermediate


## Prerequisites

**pip3 packages:**
- `azure-functions` — **not currently in the repo's root `requirements.txt`**, install with:
  ```bash
  pip3 install azure-functions
  ```
  (This folder's own `requirements.txt` already lists it — Azure Functions projects conventionally carry their
  own `requirements.txt`, separate from this repo's root one, since each Function App is deployed independently.)

**Other tooling required (not pip3 packages):**
- **Azure Functions Core Tools** (the `func` CLI) — needed to run this app locally. On macOS, this is
  typically installed via Homebrew (`brew tap azure/functions && brew install azure-functions-core-tools@4`);
  see Microsoft's official Azure Functions Core Tools documentation for the current installation method for
  your OS.
- **Azurite** or another Azure Storage emulator — `local.settings.json` sets
  `"AzureWebJobsStorage": "UseDevelopmentStorage=true"`, which points the Functions runtime at a local storage
  emulator rather than a real Storage Account (HTTP-triggered functions don't strictly need storage to run,
  but the Functions host itself expects this setting to be resolvable).

**Azure resource(s) required for a real deployment:** An **Azure Function App** resource (Consumption,
Premium, or Dedicated hosting plan) plus a Storage Account — not needed to run this locally with `func start`.

**Configuration file:** Azure Functions apps do **not** use this repo's root `.env` / `python-dotenv` pattern —
their configuration lives in **`local.settings.json`** (already present in this folder, untouched) for local
development, and in the Function App's **Application Settings** in Azure once deployed. This file already
exists in `11. cloudxeus-func/` and needs no changes for this simple, secret-free example.


## What You'll Learn

- The Azure Functions **Python v2 programming model**: `app = func.FunctionApp()` plus `@app.route(...)`
  decorators, instead of the older `function.json` + `__init__.py` per-function folder layout
- HTTP trigger concepts: `route`, `methods`, `auth_level`, route parameters (`{order_id}`), and building a
  `func.HttpResponse`
- Why `auth_level=func.AuthLevel.ANONYMOUS` means "no key required" — convenient for local testing, but a
  real security consideration once deployed (see the `12.` variant)
- How this app's `cloudxeus_spec.json` OpenAPI document is what lets an Azure AI Foundry agent call this API as
  a custom tool
- How to actually run and exercise an Azure Function locally, since a notebook can't host an HTTP server itself


### How to actually run this locally

Since this is an HTTP-triggered Azure Function, the *real* way to execute it is:

```bash
cd "14-azure-ai103/02. Section Code/11. cloudxeus-func"
pip3 install -r requirements.txt
func start
```

`func start` reads `host.json` and `local.settings.json`, discovers the `app = func.FunctionApp()` instance in
`function_app.py`, and starts a local HTTP server (by default at `http://localhost:7071`). You'd then test it
with `curl` or Postman:

```bash
curl http://localhost:7071/api/orders
curl http://localhost:7071/api/orders/1001
curl http://localhost:7071/api/orders/9999   # expect a 404
```

The code cells below are the real, valid `function_app.py` source — annotated top to bottom — even though this
notebook itself won't serve HTTP traffic when you "run" the cells.

- **💡 Exam tip:** AI-102 doesn't test Azure Functions development in depth, but it does expect you to
  recognize **HTTP triggers** as one of several Azure Functions trigger types (others include Timer, Blob,
  Queue, Cosmos DB triggers), and to understand why a Function App is a common backend for exposing custom
  logic that an Azure AI Foundry agent's **OpenAPI tool** or **custom `FunctionTool`** can call.
- **🔄 Alternatives:** Instead of `func start` locally, you could deploy this Function App to Azure (`func
  azure functionapp publish <app-name>`) and test against its live `https://<app-name>.azurewebsites.net/api/...`
  URL — which is what `cloudxeus_spec.json`'s `servers` entry (`https://fn-cloudxeus-dev-eus-01.azurewebsites.net/api`)
  already points to.


### App initialization and sample data

`func.FunctionApp()` is the Python v2 model's central object — every `@app.route(...)`-decorated function
below is registered against this single `app` instance, and the Functions host discovers them all from this
one file. `ORDERS` is a plain in-memory dictionary standing in for a real database — in a production system
this would be replaced with a call to Cosmos DB, SQL, or another persistent store.

- **💡 Exam tip:** The Python v2 model (`func.FunctionApp()` + decorators in one file) replaced the older v1
  model (a `function.json` binding-config file plus a separate `__init__.py` per function, in per-function
  folders). AI-102 material referencing "Functions" may still show either model — recognize both, but the v2
  decorator style shown here is the current recommended approach.
- **🔄 Alternatives:** Real order data would come from a database binding (e.g. a Cosmos DB input binding) or
  an explicit SDK call inside the function body, rather than a hardcoded dict — hardcoding here keeps the demo
  self-contained and dependency-free.


In [ ]:
import azure.functions as func
import json

app = func.FunctionApp()

ORDERS = {
    "1001": {
        "order_id": "1001",
        "customer": "Northgate Retail",
        "status": "shipped",
        "total": 2450.00,
    },
    "1002": {
        "order_id": "1002",
        "customer": "Aurora Logistics",
        "status": "processing",
        "total": 815.50,
    },
    "1003": {
        "order_id": "1003",
        "customer": "Brightline Media",
        "status": "delivered",
        "total": 129.99,
    },
}


### `GET /orders` — list all orders

`@app.route(route="orders", methods=["GET"], auth_level=func.AuthLevel.ANONYMOUS)` registers this function to
handle `GET` requests to `/api/orders` (the Functions host prefixes routes with `/api` by default) with no
authentication required. The function receives a `func.HttpRequest` (unused here, since this endpoint takes
no parameters) and returns a `func.HttpResponse` with a JSON-serialized list of all orders and an explicit
`mimetype="application/json"` so clients know how to parse the body.

- **💡 Exam tip:** `auth_level` is one of three values: `ANONYMOUS` (no key needed — used here), `FUNCTION`
  (a per-function or host key required, via `x-functions-key` header or `?code=` query param — used in the
  `12.` variant), or `ADMIN` (the master key, broader access). AI-102 expects you to know these three levels
  and when each is appropriate.
- **🔄 Alternatives:** Pagination (`?limit=`/`?offset=` query params via `req.params`) would be a natural
  addition for a real orders list endpoint with many records, instead of always returning everything.


In [ ]:
@app.route(route="orders", methods=["GET"],
           auth_level=func.AuthLevel.ANONYMOUS)
def list_orders(req: func.HttpRequest) -> func.HttpResponse:
    return func.HttpResponse(
        json.dumps(list(ORDERS.values())),
        mimetype="application/json",
    )


### `GET /orders/{order_id}` — get a single order

The `{order_id}` segment in `route="orders/{order_id}"` is a **route parameter**, extracted at request time via
`req.route_params.get("order_id")`. If no order matches, the function returns a `404` with a JSON error body
via the `status_code=404` argument to `func.HttpResponse`; otherwise it returns the matching order as JSON.

- **💡 Exam tip:** Route parameters (`{order_id}`) are how Azure Functions HTTP triggers implement path-based
  routing similar to typical REST framework conventions — `cloudxeus_spec.json`'s corresponding OpenAPI path
  `/orders/{order_id}` documents this exact same parameter for tools/consumers that read the spec, including
  an Azure AI Foundry agent configured with this API as an OpenAPI tool.
- **🔄 Alternatives:** Query-string based lookup (`/orders?order_id=1001`, read via `req.params.get("order_id")`)
  is a common alternative to path parameters, though path parameters are the more RESTful convention for
  identifying a specific resource — and it's what this API's OpenAPI spec documents.


In [ ]:
@app.route(route="orders/{order_id}", methods=["GET"],
           auth_level=func.AuthLevel.ANONYMOUS)
def get_order(req: func.HttpRequest) -> func.HttpResponse:
    order_id = req.route_params.get("order_id")
    order = ORDERS.get(order_id)

    if order is None:
        return func.HttpResponse(
            json.dumps({"error": f"Order {order_id} not found"}),
            status_code=404,
            mimetype="application/json",
        )

    return func.HttpResponse(
        json.dumps(order),
        mimetype="application/json",
    )


## Summary

This notebook documents a two-endpoint Azure Function App — `GET /orders` and `GET /orders/{order_id}` —
served with anonymous auth, backed by an in-memory `ORDERS` dict. Run for real via `func start` and tested
with `curl`, it behaves like any small REST API. Its real purpose in this section is to be a backend an Azure
AI Foundry agent can call through an **OpenAPI custom tool**, using `cloudxeus_spec.json` as the tool
definition — the natural next step after `FunctionTool` (`05`) for exposing *server-executed* custom actions
instead of *client-executed* ones.


## Try It Yourself

1. **Easy:** Run `func start` locally and hit all three example `curl` commands above (existing order,
   missing order, list) to confirm the behavior described in this notebook matches reality.
2. **Intermediate:** Add a `POST /orders` route that accepts a JSON body and adds a new order to `ORDERS`
   (note: since `ORDERS` is in-memory, this would reset on every restart — a real API would persist to
   storage).
3. **Advanced:** Register `cloudxeus_spec.json` as an OpenAPI-based custom tool on a new Foundry
   `PromptAgentDefinition` (research the relevant tool type in `azure.ai.projects.models`, e.g. an OpenAPI
   tool class) so an agent can answer "what's the status of order 1002?" by calling this API directly, and
   compare that flow to the manual `FunctionTool` loop in `07_helpdesk_client.py`.
